---
title: "Sprint 2: Algorytmy ANN"
subtitle: "Ewaluacja metod wyszukiwania dla problemu Particle Tracking"
author: ""
date: today
format: 
  pdf:
    toc: true
    number-sections: true
    colorlinks: true
    fig-format: pdf
    fig-width: 8
    fig-height: 5
execute:
  echo: false
  warning: false
---

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
import gc
from pathlib import Path
import numpy as np
import pandas as pd
import faiss
import warnings
from sklearn.decomposition import PCA

warnings.filterwarnings('ignore')

notebook_dir = Path().resolve()
project_root = notebook_dir.parent
src_path = str(project_root / "src")

if src_path not in sys.path:
    sys.path.append(src_path)

from hep_tracking.dataset import TrackDataset
from hep_tracking.config import KNNModelConfig
from hep_tracking.models import BaseKNN, ScipyCKDTree, FaissIVFFlat, FaissIVFPQ, HnswGraph
from hep_tracking.benchmark import ANNBenchmarkRunner
from hep_tracking.plots import (
    plot_pareto_frontier, plot_scaling, 
    plot_metric_dimension_heatmap, plot_metric_lines_by_dimension
)

print("Zależności załadowane pomyślnie!")

# --- Definicja pomocniczego wrappera dla dokładnego k-NN w FAISS ---
class FaissExact(BaseKNN):
    def __init__(self, use_gpu: bool = False):
        self.use_gpu = use_gpu
        self.index = None

    def fit(self, X: np.ndarray) -> None:
        features_contig = np.ascontiguousarray(X, dtype=np.float32)
        cpu_index = faiss.IndexFlatL2(features_contig.shape[1])
        if self.use_gpu:
            res = faiss.StandardGpuResources()
            self.index = faiss.index_cpu_to_gpu(res, 0, cpu_index)
        else:
            self.index = cpu_index
        self.index.add(features_contig)

    def kneighbors(self, X: np.ndarray, k: int) -> tuple[np.ndarray, np.ndarray]:
        features_contig = np.ascontiguousarray(X, dtype=np.float32)
        distances, indices = self.index.search(features_contig, k + 1)
        if self.use_gpu:
            faiss.StandardGpuResources().syncDefaultStreamCurrentDevice()
        return distances[:, 1:], indices[:, 1:]

# Wprowadzenie
Celem drugiego sprintu projektu jest ocena wydajności algorytmów przybliżonego wyszukiwania najbliższych sąsiadów (ANN) w kontekście rekonstrukcji torów cząstek. Ze względu na ograniczenia sprzętowe akceleratorów (GPU) narzucające długość wektora $m \in \{1, 2, 3, 4, 8, 12...\}$, 5-wymiarowe dane wejściowe zostały rozszerzone do 8 wymiarów przy pomocy bezstratnej techniki Zero-Padding.

# Poszukiwanie Granicy Pareto (Recall vs QPS)
W pierwszym eksperymencie przeprowadzono przeszukiwanie siatki hiperparametrów (Grid Search) dla trzech głównych algorytmów: **FAISS IVFFlat**, **FAISS IVFPQ** (z Product Quantization) oraz **HNSW**. 

Celem było wyznaczenie kompromisu między czułością modelu (Recall) a jego przepustowością (Queries Per Second).

In [ ]:
data_dir = project_root / "data"
dataset_hard_100k = TrackDataset.from_npz(data_dir / "dataset_hard_100k.npz")
dataset_8d = TrackDataset(dataset_hard_100k.get_padded_features(8), dataset_hard_100k.y, dataset_hard_100k.event_ids)

print(f"Dane gotowe: {dataset_8d.X.shape}")

exact_knn = ScipyCKDTree(workers=-1)
exact_knn.fit(dataset_8d.X)
_, true_indices_8d = exact_knn.kneighbors(dataset_8d.X, k=5)

pareto_configs = []
for nprobe in [1, 2, 5, 10, 20, 50]:
    pareto_configs.append(KNNModelConfig(f"IVFFlat (np={nprobe})", FaissIVFFlat, {"nlist": 100, "nprobe": nprobe, "use_gpu": True}))
for nprobe in [1, 2, 5, 10, 20, 50, 100]:
    pareto_configs.append(KNNModelConfig(f"IVFPQ (np={nprobe})", FaissIVFPQ, {"nlist": 100, "m": 8, "nbits": 8, "nprobe": nprobe, "use_gpu": True}))
for ef in [10, 20, 50, 100, 200]:
    pareto_configs.append(KNNModelConfig(f"HNSW (ef={ef})", HnswGraph, {"m": 16, "ef_construction": 200, "ef": ef}))

runner = ANNBenchmarkRunner(k_neighbors=5, warmup_runs=1, num_runs=3)
df_pareto = runner.run(
    models_configs=pareto_configs,
    datasets={"hard_100k_8D": dataset_8d},
    ground_truth={"hard_100k_8D": true_indices_8d}
)

df_pareto["Dataset"] = df_pareto["Model"].apply(lambda x: x.split(" (")[1][:-1])
df_pareto["Model"] = df_pareto["Model"].apply(lambda x: x.split(" ")[0])

plot_pareto_frontier(df_pareto, title="Wydajność algorytmów ANN: Recall vs. QPS (Przestrzeń 8D z Zero-Paddingiem)")

# Analiza Wąskiego Gardła: CPU vs GPU (Crossover N)
Karty graficzne oferują potężną moc zrównoleglonych obliczeń, jednak posiadają istotny narzut czasowy związany z transferem danych przez magistralę PCIe oraz kompilacją kerneli JIT. 

Poniższy eksperyment wyznacza punkt przecięcia (Crossover Point), w którym rozmiar zbioru danych $N$ staje się na tyle duży, że użycie GPU staje się uzasadnione czasowo.


In [ ]:
target_sizes = {"1k": 1000, "10k": 10000, "100k": 100000, "1M": 1000000}
micro_sizes = [10, 100, 200, 500, 1000]

crossover_configs = [
    KNNModelConfig("CPU (IVFFlat)", FaissIVFFlat, {"nprobe": 5, "use_gpu": False}),
    KNNModelConfig("GPU (IVFFlat)", FaissIVFFlat, {"nprobe": 5, "use_gpu": True})
]

all_crossover_results = []
dataset_1M = TrackDataset.from_npz(data_dir / "dataset_hard_1M.npz")
dataset_1M_8d_X = dataset_1M.get_padded_features(8)

print("=== WYSZUKIWANIE PUNKTU PRZECIĘCIA (CROSSOVER N): CPU vs GPU ===")

for n_points in micro_sizes + list(target_sizes.values()):
    X_subset = dataset_1M_8d_X[:n_points]
    subset_dataset = TrackDataset(X_subset, dataset_1M.y[:n_points], dataset_1M.event_ids[:n_points])
    
    nlist = min(100, int(np.sqrt(n_points)))
    for cfg in crossover_configs:
        cfg.model_kwargs["nlist"] = nlist # Dynamiczna modyfikacja nlist pod N
    
    dummy_gt = np.zeros((n_points, 5), dtype=np.int64) 
    
    df_size = runner.run(crossover_configs, {f"N={n_points}": subset_dataset}, {f"N={n_points}": dummy_gt})
    df_size["Size"] = n_points
    all_crossover_results.append(df_size)

df_crossover = pd.concat(all_crossover_results, ignore_index=True)

plot_scaling(
    df_crossover[df_crossover["Size"] >= 1000], x_col="Size", y_col="Query_Time_s", 
    title="Crossover N (Makra): CPU vs GPU"
)

plot_scaling(
    df_crossover[df_crossover["Size"] <= 1000], x_col="Size", y_col="Query_Time_s", 
    title="Crossover N (Mikro): CPU vs GPU"
)

# Skalowalność Metod Przybliżonych
Wykorzystując optymalne parametry odczytane z granicy Pareto (gwarantujące czułość na poziomie ok. 95%), przeprowadzono badanie skalowalności czasowej algorytmów przybliżonych w funkcji rosnącego rozmiaru zbioru danych ($10^3 \le N \le 10^6$).

In [ ]:
scaling_configs = [
    KNNModelConfig("Exact_CPU", FaissExact, {"use_gpu": False}),
    KNNModelConfig("Exact_GPU", FaissExact, {"use_gpu": True}),
    KNNModelConfig("IVFFlat_GPU", FaissIVFFlat, {"nprobe": 20, "use_gpu": True}),
    KNNModelConfig("IVFPQ_GPU", FaissIVFPQ, {"m": 8, "nprobe": 20, "use_gpu": True}),
    KNNModelConfig("HNSW_CPU", HnswGraph, {"m": 16, "ef": 20})
]

all_scaling_results = []
print("=== START EKSPERYMENTU: TIME vs N ===")

for size_label, n_points in target_sizes.items():
    filename = data_dir / f"dataset_hard_{size_label}.npz"
    if not filename.exists():
        continue
        
    ds = TrackDataset.from_npz(filename)
    ds_8d = TrackDataset(ds.get_padded_features(8), ds.y, ds.event_ids)
    
    for cfg in scaling_configs:
        if "IVF" in cfg.name:
            cfg.model_kwargs["nlist"] = min(100, int(np.sqrt(n_points)))

    dummy_gt = np.zeros((n_points, 5), dtype=np.int64) 
    df_size = runner.run(scaling_configs, {size_label: ds_8d}, {size_label: dummy_gt})
    df_size["Size"] = n_points
    all_scaling_results.append(df_size)

df_scaling = pd.concat(all_scaling_results, ignore_index=True)

plot_scaling(
    df_scaling, x_col="Size", y_col="Query_Time_s", 
    title="Przestrzeń 8D: Algorytmy Exact vs Przybliżone ANN"
)

# Weryfikacja Założeń: Exact kNN vs ANN w Niskich Wymiarach
Ostatni etap analizy to weryfikacja celowości stosowania algorytmów heurystycznych (ANN) w przestrzeniach o bardzo małej liczbie wymiarów (5D). Do zestawienia dołączono precyzyjne metody `FAISS ExactL2` oraz strukturę podziału przestrzeni `Scipy cKDTree` (100% Recall).

In [ ]:
low_dim_configs = [
    KNNModelConfig("Exact_CPU (Faiss)", FaissExact, {"use_gpu": False}),
    KNNModelConfig("Exact_GPU (Faiss)", FaissExact, {"use_gpu": True}),
    KNNModelConfig("cKDTree_CPU", ScipyCKDTree, {"workers": -1}),
    KNNModelConfig("IVFFlat_GPU", FaissIVFFlat, {"nprobe": 20, "use_gpu": True}),
    KNNModelConfig("HNSW_CPU", HnswGraph, {"m": 16, "ef": 20})
]

all_low_dim_results = []
print("=== START EKSPERYMENTU: EXACT kNN vs ANN (Przestrzeń 5D) ===")

for size_label, n_points in {"10k": 10000, "100k": 100000, "1M": 1000000}.items():
    filename = data_dir / f"dataset_hard_{size_label}.npz"
    if not filename.exists():
        continue
        
    ds_5d = TrackDataset.from_npz(filename)
    actual_n = len(ds_5d)
    
    for cfg in low_dim_configs:
        if "IVF" in cfg.name:
            cfg.model_kwargs["nlist"] = min(100, int(np.sqrt(actual_n)))

    dummy_gt = np.zeros((actual_n, 5), dtype=np.int64) 
    df_size = runner.run(low_dim_configs, {size_label: ds_5d}, {size_label: dummy_gt})
    df_size["Size"] = actual_n
    all_low_dim_results.append(df_size)

df_low_dim = pd.concat(all_low_dim_results, ignore_index=True)

plot_scaling(
    df_low_dim, x_col="Size", y_col="Query_Time_s", 
    title="Wymiar 5D: Exact kNN vs ANN"
)

## Wnioski Końcowe (Curse of Dimensionality)

Wyniki powyższego starcia dostarczają kluczowych wniosków inżynieryjnych:

1. **Deklasacja przez metody oparte na drzewach:** W przestrzeni 5-wymiarowej matematycznie dokładny algorytm `cKDTree` (złożoność $O(N \log N)$) deklasuje zaawansowane metody przybliżone.
2. **Metody ANN:** Z metod przybliżonych ANN IVFFlat zdaje się radzić sobie najlepiej. Możliwe, że dla zbiorów danych o większej liczebności lub liczbie wymiarów uzyskałby wyniki lepsze od cKDTree 


# Eksperyment: Czas w funkcji wymiarowości (D) i rozmiaru danych (N)

Poprzednie eksperymenty badały skalowanie względem N przy ustalonym D (5D lub 8D po
Zero-Paddingu). Tutaj obie zmienne zmieniają się jednocześnie: D ∈ {2, 4, 8} oraz
N ∈ {1k, 10k, 100k, 1M}.

Dla D=2 i D=4 wykorzystywane są pierwsze D kolumn oryginalnych 5-wymiarowych cech
(obcięcie — dopełnianie zerami "w dół" nie miałoby sensu fizycznego). Dla D=8
zachowana jest metodologia Zero-Padding z poprzednich sekcji. Dla IVFPQ liczba
subkwantyzatorów `m` jest zawsze równa D (musi dzielić D bez reszty), analogicznie
do sekcji "Granica Pareto", gdzie dla D=8 użyto m=8.

In [ ]:
dims_to_test = [2, 4, 8]
all_dim_results = []

print("=== EKSPERYMENT: CZAS i RECALL vs WYMIAROWOŚĆ (D) x ROZMIAR DANYCH (N) ===")

exact_knn = ScipyCKDTree(workers=-1)

for size_label, n_points in target_sizes.items():
    filename = data_dir / f"dataset_hard_{size_label}.npz"
    if not filename.exists():
        continue

    ds_original = TrackDataset.from_npz(filename)
    actual_n = len(ds_original)
    nlist = min(100, int(np.sqrt(actual_n)))

    exact_knn.fit(ds_original.X)
    _, true_indices_5d = exact_knn.kneighbors(ds_original.X, k=5)

    for d in dims_to_test:
        if d < ds_original.X.shape[1]:
            pca = PCA(n_components=d, random_state=42)
            X_d = pca.fit_transform(ds_original.X).astype(np.float32)
        elif d == ds_original.X.shape[1]:
            X_d = ds_original.X
        else:
            X_d = ds_original.get_padded_features(target_dim=d)
            
        ds_d = TrackDataset(X=X_d, y=ds_original.y, event_ids=ds_original.event_ids)
        
        dim_configs = [
            KNNModelConfig("IVFFlat", FaissIVFFlat, {"nlist": nlist, "nprobe": 20, "use_gpu": True}),
            KNNModelConfig("IVFPQ", FaissIVFPQ, {"nlist": nlist, "m": d, "nprobe": 20, "use_gpu": True}),
            KNNModelConfig("HNSW", HnswGraph, {"m": 16, "ef": 20})
        ]
        
        df_run = runner.run(dim_configs, {"DS": ds_d}, {"DS": true_indices_5d})
        df_run["Size"] = actual_n
        df_run["Dimension"] = d
        all_dim_results.append(df_run)

df_dims = pd.concat(all_dim_results, ignore_index=True)

plot_metric_dimension_heatmap(df_dims, metric="Recall", title="Recall: Wymiarowość (D) vs Rozmiar danych (N)", cmap="RdYlGn")
plot_metric_lines_by_dimension(df_dims, metric="Recall", title="Recall w funkcji N, dla różnych wymiarowości D")

plot_metric_dimension_heatmap(df_dims, metric="Query_Time_s", title="Czas zapytania: Wymiarowość (D) vs Rozmiar danych (N)", log_scale=True)
plot_metric_lines_by_dimension(df_dims, metric="Query_Time_s", title="Czas zapytania w funkcji N", log_y=True)

 Wyniki: Recall w funkcji wymiarowości (D) i rozmiaru danych (N)
 
## Recall — mapa cieplna
 
| D \ N | 1 000 | 10 000 | 100 000 | 1 000 000 |
|---|---|---|---|---|
| **IVFFlat** 2D/4D/8D | 1.000 | 1.000 | 1.000 | 1.000 |
| **IVFPQ** 2D | 0.987 | 0.972 | 0.927 | 0.764 |
| **IVFPQ** 4D | 0.994 | 0.983 | 0.966 | 0.939 |
| **IVFPQ** 8D | 0.987 | 0.984 | 0.971 | 0.948 |
| **HNSW** 2D/4D/8D | 1.000 | 1.000 | 1.000 | 1.000 |
 
## Wnioski
 
**1. IVFFlat i HNSW osiągają idealny recall (1.000) niezależnie od D i N.**
IVFFlat liczy dokładną odległość L2 wewnątrz wybranych klastrów (bez kwantyzacji wektorów),
a HNSW przy `ef=20` na tej skali danych praktycznie nie traci sąsiadów. Oba wyniki są
stabilne i przewidywalne w całym testowanym zakresie.
 
**2. Jedynym algorytmem tracącym recall jest IVFPQ — i tylko on.** To, że pozostałe dwa
algorytmy pozostają idealne, potwierdza, że dane są teraz poprawne (wcześniejszy problem
z identycznym spadkiem recall we wszystkich trzech algorytmach jednocześnie wynikał
z duplikacji zdarzeń w generatorze danych — po naprawie `seed` ten efekt zniknął).
 
**3. IVFPQ degraduje silniej wraz ze wzrostem N, ale słabiej wraz ze wzrostem D — to
wynik konfiguracji `m = D`.** Każdy sub-kwantyzator odpowiada wtedy dokładnie za jedną
współrzędną. Błąd kwantyzacji jednej współrzędnej to stały, bezwzględny szum:
- przy D=2 ta jedna współrzędna to połowa informacji o pozycji punktu → silny wpływ na ranking odległości (recall spada do 0.764 przy N=1M),
- przy D=8 ten sam bezwzględny szum rozkłada się na osiem współrzędnych przy sumowaniu odległości → wpływ względny jest znacznie mniejszy (recall = 0.948 przy N=1M).
